# How many replicates does a check need?

Reads the ten replicates of one configuration written by

```bash
python experiments/exploration-replicate-variability/run.py
```

`replication-reporting`, ten examples from ten distinct documents, ten
identical runs. The note is
[`thinking/experiments/exploration-replicate-variability.md`](../../thinking/experiments/exploration-replicate-variability.md)
— read it first: it records that this probe aims deliberately at the noisiest
check, so the number here is closer to an upper bound than a typical value.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from soda_mmqc import cli

RUNS = Path("../../experiments/runs/exploration-replicate-variability").resolve()
CHECKLIST, CHECK, MODEL = "fig-checklist-exp01", "replication-reporting", "claude-sonnet-5"
FULL_BENCHMARK = 38   # examples exp-01 will run; this probe uses ten of them

reps = sorted((RUNS / "pinned").glob("rep-*"))
assert reps, f"no replicates under {RUNS}; run the probe first"
[p.name for p in reps]

## One score per example per replicate

`score_check` takes one `rep-NN/`. Ten calls give a 10 × 10 grid: the same
examples, measured ten times.

In [ ]:
rows = []
for rep in reps:
    index = int(rep.name.split("-")[1])
    result = cli.score_check(CHECKLIST, CHECK, rep, model=MODEL, save=False)
    for record in result["agentic"]["flat"]:
        props = (record["analysis"].get("by_property") or {})
        means = [s.get("mean_score") for s in props.values() if s.get("mean_score") is not None]
        rows.append({
            "replicate": index,
            "example": record.get("doc_id"),
            "score": float(np.mean(means)) if means else np.nan,
        })

grid = pd.DataFrame(rows).pivot(index="example", columns="replicate", values="score")
print(f"{grid.shape[0]} examples x {grid.shape[1]} replicates")
grid.round(3)

## The decomposition

The quantity exp-01 needs is the spread of a **check mean over 38 examples**.
This probe has ten, so the observed spread must not be read as exp-01's — a
mean over ten examples is noisier by about √(38/10) ≈ 2.

Instead estimate the *per-example* between-replicate variance, which does not
depend on how many examples were run, and derive the check-level spread from
it:

$$\mathrm{SD}(\bar{s}_N) = \sqrt{\bar{v}/N}, \qquad \bar{v} = \text{mean per-example variance}$$

In [ ]:
per_example_var = grid.var(axis=1, ddof=1)      # variance across replicates, per example
v_bar = per_example_var.mean()

print("per-example SD across replicates:")
print(np.sqrt(per_example_var).round(4).to_string())
print(f"\nmean per-example variance : {v_bar:.6f}")
print(f"mean per-example SD       : {np.sqrt(v_bar):.4f}")

### Does the extrapolation hold?

It assumes examples vary independently. Test it: the derived spread at N=10
should match the spread actually observed across the ten replicate means. If
they disagree materially, examples are correlated and the step to 38 is not
valid — say so in the note rather than using the number.

In [ ]:
observed = grid.mean(axis=0).std(ddof=1)        # SD of the ten check means, N=10
derived_10 = np.sqrt(v_bar / grid.shape[0])

print(f"observed SD of check mean (N=10) : {observed:.4f}")
print(f"derived  SD of check mean (N=10) : {derived_10:.4f}")
ratio = observed / derived_10 if derived_10 else np.nan
print(f"ratio                            : {ratio:.2f}")
print("\n" + ("independence looks reasonable" if 0.7 <= ratio <= 1.4
               else "MISMATCH -- examples are not independent; do not extrapolate"))

## What a replicate count buys

`SE(n) = SD(mean over 38) / √n`. Compare against the difference exp-01 is
looking for: if detailed-minus-minimal is smaller than `SE(5)`, five
replicates will not resolve it and the design needs changing rather than more
sampling.

In [ ]:
sd_38 = np.sqrt(v_bar / FULL_BENCHMARK)
print(f"SD of a check mean over {FULL_BENCHMARK} examples: {sd_38:.4f}\n")
pd.DataFrame({
    "replicates": [1, 3, 5, 7, 10],
    "SE of one arm": [sd_38 / np.sqrt(n) for n in (1, 3, 5, 7, 10)],
    "SE of a paired difference (upper bound)":
        [sd_38 * np.sqrt(2) / np.sqrt(n) for n in (1, 3, 5, 7, 10)],
}).round(4)

The second column is an **upper bound** on what exp-01 actually faces. Its
statistic is a difference of two arms, so if the arms varied independently
its variance would be twice one arm's. Pairing by example cancels whatever
noise is common to both, so the true value is lower — by how much, only exp-01
can say.

## Where the variance sits

If it is concentrated in one or two examples, that is a property of those
cases rather than of the model, and more replicates buy less than the average
suggests.

In [ ]:
pd.DataFrame({
    "mean": grid.mean(axis=1),
    "sd": grid.std(axis=1, ddof=1),
    "min": grid.min(axis=1),
    "max": grid.max(axis=1),
}).sort_values("sd", ascending=False).round(3)

## Non-response

A replicate that returned nothing scores as a fully missing row set rather
than being excluded. Even one changes how the average should be read.

In [ ]:
import json as _json

empty = [
    {"replicate": int(rep.name.split("-")[1]),
     "example": str(p.parent.relative_to(rep))}
    for rep in reps
    for p in sorted(rep.rglob("prediction.json"))
    if not _json.loads(p.read_text())["outputs"]
]
pd.DataFrame(empty) if empty else "no empty answers in any replicate"

## Cost

In [ ]:
usage = [
    {"usd": u["total_cost_usd"], "turns": u.get("num_turns"),
     "seconds": (u.get("duration_ms") or 0) / 1000}
    for rep in reps
    for a in sorted(rep.rglob("tool_audit.json"))
    for u in [(_json.loads(a.read_text()).get("usage") or {})]
    if u.get("total_cost_usd") is not None
]
spend = pd.DataFrame(usage)
if len(spend):
    per_session = spend["usd"].mean()
    print(f"sessions    : {len(spend)}")
    print(f"per session : ${per_session:.4f}")
    print(f"this probe  : ${spend['usd'].sum():.2f}")
    print(f"median turns/seconds: {spend['turns'].median():.0f} / {spend['seconds'].median():.1f}")
    print("\nexp-01, 11 checks x 2 arms x 436 example-checks:")
    for n in (3, 5):
        print(f"  {n} replicates: {436*2*n:>5,} sessions  ~${436*2*n*per_session:,.0f}")
else:
    print("no usage recorded -- these runs predate cost capture")

## The decision goes in the note

Not here. Record which replicate count exp-01 will use, the `SE(n)` that
justified it, and whether the independence check passed — and keep the two
caveats visible: this is one check chosen for being noisy, and one arm rather
than a paired difference.